# 00｜OpenBB环境快速构建

这个Notebook用于每次覆盖项目补丁后，快速完成以下工作：

1. 自动定位`qianji_openbb_mini`项目根目录；
2. 确认所有命令都使用当前Notebook内核对应的Python；
3. 重新安装本地`qianji-data-mini`和`openbb-choice`；
4. 执行`openbb.build()`重新发现`qianji`和`choice` Provider；
5. 使用全新的Python进程检查包版本、Provider注册和EmQuantAPI导入；
6. 导出不含凭据的JSON构建记录。

默认是`quick`快速模式，不主动升级OpenBB等运行时依赖，适合已经配置好的`dm311`环境。只有新建空环境时才改为`full`。

安全边界：本Notebook不读取`.env`，不登录Choice，也不会打印账号、密码、Token或`userInfo`内容。

## 1. 确认当前Python并定位项目

In [1]:
import os
import sys
from pathlib import Path

print("当前Python：", sys.executable)
print("Python版本：", sys.version.split()[0])
print("Conda环境：", os.getenv("CONDA_DEFAULT_ENV", "未检测到"))
print("当前目录：", Path.cwd().resolve())

# Notebook放在项目notebooks目录时无需填写。
# 若单独存放，再填写真实项目根目录；下划线前不要加反斜杠。
# 示例：PROJECT_ROOT_OVERRIDE = r"D:\OneDrive\桌面\qianji_openbb_mini"
PROJECT_ROOT_OVERRIDE = ""


def find_project_root(start: Path) -> Path:
    if PROJECT_ROOT_OVERRIDE.strip():
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate
        raise FileNotFoundError(f"指定的项目根目录不正确：{candidate}")

    for candidate in (start, *start.parents):
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "src" / "qianji_data_mini").exists()
            and (candidate / "extensions" / "openbb_choice").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "没有找到qianji_openbb_mini项目。请把本Notebook放入项目notebooks目录，"
        "或填写PROJECT_ROOT_OVERRIDE。"
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CHOICE_EXTENSION = PROJECT_ROOT / "extensions" / "openbb_choice"
OUTPUT_DIR = PROJECT_ROOT / "validation_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("项目根目录：", PROJECT_ROOT)
print("Choice插件目录：", CHOICE_EXTENSION)

当前Python： d:\minicoda3\envs\dm311\python.exe
Python版本： 3.11.14
Conda环境： dm311
当前目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\notebooks
项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
Choice插件目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\extensions\openbb_choice


## 2. 选择构建模式

In [2]:
# quick：补丁覆盖后的日常快速重装，不主动升级运行时依赖（推荐）。
# full：新建环境或确认缺少依赖时使用，可能访问软件包仓库并耗时较长。
INSTALL_MODE = "full"

# 正常情况下保持True。
REBUILD_OPENBB = True
VERIFY_PROVIDERS = True

if INSTALL_MODE not in {"quick", "full"}:
    raise ValueError("INSTALL_MODE只能是quick或full。")

print("安装模式：", INSTALL_MODE)
print("重新构建OpenBB：", REBUILD_OPENBB)
print("验证Provider：", VERIFY_PROVIDERS)

安装模式： full
重新构建OpenBB： True
验证Provider： True


## 3. 检查补丁文件和项目声明版本

In [3]:
import importlib.util
import tomllib


def read_project_version(pyproject_path: Path) -> str:
    payload = tomllib.loads(pyproject_path.read_text(encoding="utf-8"))
    return str(payload["project"]["version"])


required_paths = {
    "主项目pyproject": PROJECT_ROOT / "pyproject.toml",
    "qianji Provider": PROJECT_ROOT / "src" / "qianji_data_mini" / "openbb_provider" / "provider.py",
    "Choice插件pyproject": CHOICE_EXTENSION / "pyproject.toml",
    "Choice Provider": CHOICE_EXTENSION / "openbb_choice" / "provider.py",
    "Choice月线修复": CHOICE_EXTENSION / "openbb_choice" / "utils" / "emquant.py",
}

missing_paths = []
for label, path in required_paths.items():
    exists = path.exists()
    print(f"{label}：", "存在" if exists else f"缺失 -> {path}")
    if not exists:
        missing_paths.append(str(path))

if missing_paths:
    raise FileNotFoundError("补丁或项目文件不完整：\n" + "\n".join(missing_paths))

DECLARED_QIANJI_VERSION = read_project_version(PROJECT_ROOT / "pyproject.toml")
DECLARED_CHOICE_VERSION = read_project_version(CHOICE_EXTENSION / "pyproject.toml")

print("项目声明 qianji-data-mini：", DECLARED_QIANJI_VERSION)
print("项目声明 openbb-choice：", DECLARED_CHOICE_VERSION)

if INSTALL_MODE == "quick" and importlib.util.find_spec("openbb") is None:
    raise RuntimeError(
        "当前Python尚未安装OpenBB，不能使用quick模式。"
        "请把INSTALL_MODE改为full后重新运行本格。"
    )

主项目pyproject： 存在
qianji Provider： 存在
Choice插件pyproject： 存在
Choice Provider： 存在
Choice月线修复： 存在
项目声明 qianji-data-mini： 0.6.0
项目声明 openbb-choice： 0.1.2


## 4. 定义安全的命令执行函数

In [4]:
import locale
import subprocess
from datetime import datetime


command_records = []


def run_command(label: str, arguments: list[str], *, stop_on_error: bool = True):
    print(f"\n{'=' * 12} {label} {'=' * 12}")
    print("执行Python：", sys.executable)
    completed = subprocess.run(
        arguments,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        encoding=locale.getpreferredencoding(False) or "utf-8",
        errors="replace",
    )
    print("返回码：", completed.returncode)
    if completed.stdout.strip():
        print("标准输出：\n", completed.stdout)
    if completed.stderr.strip():
        print("错误输出：\n", completed.stderr)

    command_records.append(
        {
            "label": label,
            "returncode": completed.returncode,
            "success": completed.returncode == 0,
        }
    )
    if stop_on_error and completed.returncode != 0:
        raise RuntimeError(f"{label}失败。请保留本格显示的返回码、标准输出和错误输出。")
    return completed


run_command("检查pip", [sys.executable, "-m", "pip", "--version"])


============ 检查pip ============
执行Python： d:\minicoda3\envs\dm311\python.exe
返回码： 0
标准输出：
 pip 26.0.1 from d:\minicoda3\envs\dm311\Lib\site-packages\pip (python 3.11)




CompletedProcess(args=['d:\\minicoda3\\envs\\dm311\\python.exe', '-m', 'pip', '--version'], returncode=0, stdout='pip 26.0.1 from d:\\minicoda3\\envs\\dm311\\Lib\\site-packages\\pip (python 3.11)\n\n', stderr='')

## 5. 快速重装本地项目与Choice插件

In [5]:
if INSTALL_MODE == "quick":
    qianji_install_target = str(PROJECT_ROOT)
    choice_install_target = str(CHOICE_EXTENSION)
    common_options = ["--no-deps"]
else:
    qianji_install_target = str(PROJECT_ROOT) + "[openbb]"
    choice_install_target = str(CHOICE_EXTENSION)
    common_options = []

run_command(
    "安装qianji-data-mini",
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        *common_options,
        "-e",
        qianji_install_target,
    ],
)

run_command(
    "安装openbb-choice",
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        *common_options,
        "-e",
        choice_install_target,
    ],
)

print("两个本地包安装完成。")


============ 安装qianji-data-mini ============
执行Python： d:\minicoda3\envs\dm311\python.exe
返回码： 0
标准输出：
 Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Obtaining file:///D:/OneDrive/%E6%A1%8C%E9%9D%A2/%E6%95%B0%E6%8D%AE%E5%9F%BA%E5%BA%A7%E4%BB%A3%E7%A0%81/qianji_openbb_mini
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for qianji-data-mini (pyproject.toml): started
  Building editable for qianji-data-mini (pyproject.toml): finished with status 'done'
  Created wheel for qianji-data-mini: filename=qianji_data_mi

## 6. 重新构建OpenBB Provider注册表

In [6]:
if REBUILD_OPENBB:
    run_command(
        "执行openbb.build()",
        [sys.executable, "-c", "import openbb; openbb.build()"],
    )
else:
    print("已跳过OpenBB构建。覆盖Provider补丁后通常不应跳过。")


============ 执行openbb.build() ============
执行Python： d:\minicoda3\envs\dm311\python.exe
返回码： 0
标准输出：
 Extensions to add: qianji@0.6.0
Extensions to remove: qianji@0.5.0

Building...



## 7. 使用全新Python进程验证版本和Provider

In [7]:
import json

verification_code = r'''
import importlib.util
import json
import sys
from importlib.metadata import PackageNotFoundError, version

def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "未安装"

payload = {
    "python": sys.executable,
    "python_version": sys.version.split()[0],
    "qianji_data_mini_version": package_version("qianji-data-mini"),
    "openbb_choice_version": package_version("openbb-choice"),
    "emquant_importable": importlib.util.find_spec("EmQuantAPI") is not None,
    "providers": [],
}

from openbb import obb
payload["providers"] = sorted(set(obb.coverage.providers))
print("__QIANJI_VERIFY__" + json.dumps(payload, ensure_ascii=False))
'''

verification_run = run_command(
    "独立进程环境验收",
    [sys.executable, "-c", verification_code],
)

marker_lines = [
    line for line in verification_run.stdout.splitlines()
    if line.startswith("__QIANJI_VERIFY__")
]
if not marker_lines:
    raise RuntimeError("独立进程没有返回结构化验收结果。")

verification = json.loads(marker_lines[-1].removeprefix("__QIANJI_VERIFY__"))
providers = set(verification["providers"])

checks = {
    "使用当前Notebook Python": Path(verification["python"]).resolve() == Path(sys.executable).resolve(),
    "qianji版本与项目一致": verification["qianji_data_mini_version"] == DECLARED_QIANJI_VERSION,
    "choice版本与项目一致": verification["openbb_choice_version"] == DECLARED_CHOICE_VERSION,
    "发现qianji Provider": "qianji" in providers,
    "发现choice Provider": "choice" in providers,
    "EmQuantAPI可以导入": bool(verification["emquant_importable"]),
}

for label, passed in checks.items():
    print(f"{label}：", "PASS" if passed else "FAIL")

print("qianji-data-mini版本：", verification["qianji_data_mini_version"])
print("openbb-choice版本：", verification["openbb_choice_version"])

if VERIFY_PROVIDERS and not all(checks.values()):
    failed = [label for label, passed in checks.items() if not passed]
    raise RuntimeError("环境验收未通过：" + "、".join(failed))


============ 独立进程环境验收 ============
执行Python： d:\minicoda3\envs\dm311\python.exe
返回码： 0
标准输出：
 __QIANJI_VERIFY__{"python": "d:\\minicoda3\\envs\\dm311\\python.exe", "python_version": "3.11.14", "qianji_data_mini_version": "0.6.0", "openbb_choice_version": "0.1.2", "emquant_importable": true, "providers": ["benzinga", "bls", "cftc", "choice", "congress_gov", "econdb", "eia", "federal_reserve", "fmp", "fred", "government_us", "imf", "intrinio", "oecd", "qianji", "sec", "tiingo", "tradingeconomics", "yfinance"]}

使用当前Notebook Python： PASS
qianji版本与项目一致： PASS
choice版本与项目一致： PASS
发现qianji Provider： PASS
发现choice Provider： PASS
EmQuantAPI可以导入： PASS
qianji-data-mini版本： 0.6.0
openbb-choice版本： 0.1.2


## 8. 导出不含凭据的构建记录

In [8]:
report = {
    "generated_at": datetime.now().astimezone().isoformat(),
    "python": sys.executable,
    "python_version": sys.version.split()[0],
    "project_root": str(PROJECT_ROOT),
    "install_mode": INSTALL_MODE,
    "declared_versions": {
        "qianji-data-mini": DECLARED_QIANJI_VERSION,
        "openbb-choice": DECLARED_CHOICE_VERSION,
    },
    "verification": verification,
    "checks": checks,
    "commands": command_records,
    "credentials_included": False,
}

timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
report_path = OUTPUT_DIR / f"OpenBB环境构建_{timestamp}.json"
report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("构建记录：", report_path)
print("凭据已导出：", report["credentials_included"])

构建记录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\OpenBB环境构建_20260902_084711.json
凭据已导出： False


## 9. 完成后操作

In [9]:
if all(checks.values()):
    print("环境构建结论：全部通过。")
    print("下一步：彻底重启Notebook内核，再从头运行其它Notebook。")
else:
    print("环境构建结论：存在失败项，请查看第7格。")

环境构建结论：全部通过。
下一步：彻底重启Notebook内核，再从头运行其它Notebook。
